# Modul 16: CNNs und Transfer Learning mit Keras | Lösungen

## Überblick

Sie bereiten kleine Bildtensoren für Keras vor, planen Faltungsformen und trainieren ein LeNet-ähnliches CNN. Anschließend verwenden Sie MobileNetV2 als eingefrorenen Merkmalsextraktor, trainieren einen kleinen Klassifikationskopf und vergleichen Leistung, Größe und Laufzeit.

**Zugehörige Vorlesungen**

- **LeNet mit Keras**
- **Transfer mit Keras**

## Lernziele

Nach der Bearbeitung können Sie:

- Bildtensoren mit Kanaldimension, reproduzierbaren Splits und einheitlicher Normalisierung vorbereiten.
- ein kleines LeNet-ähnliches Keras-CNN erstellen, trainieren und anhand von Lernkurven sowie Fehlerbildern bewerten.
- MobileNetV2 ressourcenschonend für Transfer Learning einfrieren, passend vorverarbeiten und mit einer Baseline vergleichen.

## Geprüfte Fähigkeiten

- Conv2D-, Padding-, Stride-, Pooling- und Tensorformplanung
- Keras-CNN-Training, Dropout, EarlyStopping, Konfusionsmatrix und Fehlklassifikationen
- Keras Applications, MobileNetV2, preprocess_input, eingefrorene Basis und Transfer-Kopf

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt den kleinen Digits-Datensatz aus scikit-learn. Die 8-mal-8-Graustufenbilder werden in Training, Validierung und Test geteilt. LeNet arbeitet direkt mit den kleinen Bildern. Für MobileNetV2 werden nur kleine Teilmengen auf 64-mal-64 RGB skaliert.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

digits = load_digits()
bilder_gesamt = digits.images.astype("float32")
labels_gesamt = digits.target.astype("int64")

bilder_train_roh, bilder_test_roh, labels_train, labels_test = train_test_split(
    bilder_gesamt,
    labels_gesamt,
    test_size=0.20,
    stratify=labels_gesamt,
    random_state=RANDOM_SEED,
)
bilder_train_roh, bilder_val_roh, labels_train, labels_val = train_test_split(
    bilder_train_roh,
    labels_train,
    test_size=0.20,
    stratify=labels_train,
    random_state=RANDOM_SEED,
)

print("Train, Validierung, Test:", bilder_train_roh.shape, bilder_val_roh.shape, bilder_test_roh.shape)
print("TensorFlow-Version:", tf.__version__)

### Aufgabe 1: Bildtensoren normalisieren und Kanäle ergänzen

Skalieren Sie alle Pixelwerte mit derselben festen Regel von 0 bis 16 auf 0 bis 1 und ergänzen Sie eine Kanaldimension. Die resultierende Form soll `(Anzahl, 8, 8, 1)` sein.

Prüfen Sie Datentyp, Wertebereich und Formen aller Splits. Visualisieren Sie fünf Trainingsbilder mit ihren Labels und erklären Sie, warum die Testdaten keine eigene Normalisierungsregel lernen dürfen.

In [ ]:
# Die Digits-Pixelwerte liegen laut Datensatzbeschreibung zwischen 0 und 16.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def bereite_digit_bilder_vor(bilder):
    """Skaliert Pixel fest auf [0, 1] und ergänzt die Graustufen-Kanalachse."""
    tensor = np.asarray(bilder, dtype="float32") / 16.0
    return tensor[..., np.newaxis]

bilder_train = bereite_digit_bilder_vor(bilder_train_roh)
bilder_val = bereite_digit_bilder_vor(bilder_val_roh)
bilder_test = bereite_digit_bilder_vor(bilder_test_roh)

for name, teil in {
    "Training": bilder_train,
    "Validierung": bilder_val,
    "Test": bilder_test,
}.items():
    print(
        name,
        "Form=", teil.shape,
        "dtype=", teil.dtype,
        "Minimum=", float(teil.min()),
        "Maximum=", float(teil.max()),
    )
    assert teil.ndim == 4 and teil.shape[-1] == 1
    assert 0.0 <= teil.min() <= teil.max() <= 1.0

for index in range(5):
    plt.figure()
    plt.imshow(bilder_train[index, ..., 0], cmap="gray", vmin=0, vmax=1)
    plt.title(f"Trainingslabel: {labels_train[index]}")
    plt.axis("off")
    plt.show()

> **Musterantwort und Interpretation**
>
> Der gültige Pixelbereich ist durch den Datensatz bekannt. Deshalb kann dieselbe deterministische Transformation ohne Schätzung auf Training, Validierung und Test angewendet werden. Bei datenabhängigen Transformationen, etwa Mittelwert und Standardabweichung, müssten die Parameter ausschließlich aus dem Training gelernt werden.

### Aufgabe 2: Faltungsformen und Feature Maps kontrollieren

Erstellen Sie eine einzelne `Conv2D`-Schicht mit vier Filtern der Größe 3-mal-3, ReLU, Stride 1 und `padding="same"`. Wenden Sie sie auf einen Batch von acht Bildern an. Führen Sie danach 2-mal-2-MaxPooling aus.

Geben Sie alle Formen aus und berechnen Sie die erwartete Höhe und Breite zusätzlich mit einer eigenen Formel. Visualisieren Sie die vier Feature Maps des ersten Bildes. Die Filter sind noch untrainiert, daher ist nur die Form, nicht die inhaltliche Qualität zu interpretieren.

In [ ]:
def ausgabelaenge(eingabe, kernel, stride=1, padding=0):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def ausgabelaenge(eingabe, kernel, stride=1, padding=0):
    return int(np.floor((eingabe + 2 * padding - kernel) / stride) + 1)

conv_demo = layers.Conv2D(
    filters=4,
    kernel_size=3,
    strides=1,
    padding="same",
    activation="relu",
)
pool_demo = layers.MaxPooling2D(pool_size=2, strides=2)

mini_batch = tf.convert_to_tensor(bilder_train[:8])
feature_maps = conv_demo(mini_batch)
gepoolte_maps = pool_demo(feature_maps)

print("Eingabe:", mini_batch.shape)
print("Nach Conv2D:", feature_maps.shape)
print("Nach MaxPooling:", gepoolte_maps.shape)

# Bei einem ungeraden 3er-Kernel und same-Padding wird effektiv je Seite ein Nullrand genutzt.
erwartete_conv = ausgabelaenge(8, kernel=3, stride=1, padding=1)
erwartete_pool = ausgabelaenge(erwartete_conv, kernel=2, stride=2, padding=0)
assert feature_maps.shape[1:3] == (erwartete_conv, erwartete_conv)
assert gepoolte_maps.shape[1:3] == (erwartete_pool, erwartete_pool)

for filter_index in range(4):
    plt.figure()
    plt.imshow(feature_maps[0, ..., filter_index].numpy(), cmap="gray")
    plt.title(f"Untrainierte Feature Map {filter_index + 1}")
    plt.axis("off")
    plt.show()

> **Musterantwort und Interpretation**
>
> Conv2D bestimmt durch Filterzahl die Kanaltiefe und durch Kernel, Stride und Padding die räumliche Größe. ReLU verändert Werte, aber nicht die Tensorform. MaxPooling reduziert typischerweise Höhe und Breite, während die Anzahl der Kanäle erhalten bleibt.

### Aufgabe 3: Ein LeNet-ähnliches Keras-CNN aufbauen

Erstellen Sie eine Funktion `baue_lenet(use_dropout=False)`. Das Modell soll zwei Faltungsblöcke aus Conv2D, ReLU und MaxPooling enthalten, danach Flatten, eine kleine Dense-Schicht und zehn Softmax-Ausgaben.

Nutzen Sie höchstens 16 beziehungsweise 32 Filter und höchstens 64 Dense-Einheiten. Optional soll vor der Ausgabeschicht Dropout mit Rate 0.25 eingefügt werden. Kompilieren Sie mit Adam, sparse categorical crossentropy und Accuracy. Prüfen Sie Ausgabeform und Parameterzahl.

In [ ]:
def baue_lenet(use_dropout=False):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def baue_lenet(use_dropout=False):
    """Erzeugt ein kleines, für 8-mal-8-Digits angepasstes LeNet-Modell."""
    modell_schichten = [
        keras.Input(shape=(8, 8, 1)),
        layers.Conv2D(16, kernel_size=3, padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
    ]
    if use_dropout:
        modell_schichten.append(layers.Dropout(0.25, seed=RANDOM_SEED))
    modell_schichten.append(layers.Dense(10, activation="softmax"))

    modell = keras.Sequential(modell_schichten, name="kleines_lenet")
    modell.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modell

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_SEED)
lenet_modell = baue_lenet(use_dropout=True)
lenet_modell.summary()

test_ausgabe = lenet_modell(bilder_train[:4], training=False)
print("Ausgabeform:", test_ausgabe.shape)
print("Zeilensummen:", tf.reduce_sum(test_ausgabe, axis=1).numpy())
print("Parameterzahl:", lenet_modell.count_params())
assert test_ausgabe.shape == (4, 10)

> **Musterantwort und Interpretation**
>
> Der Datensatz enthält genau die zehn gegenseitig ausschließenden Klassen 0 bis 9. Jede Ausgabeeinheit repräsentiert eine Klassenwahrscheinlichkeit, und Softmax normiert die zehn Werte pro Bild auf Summe eins. Sparse categorical crossentropy passt dazu, weil die Zielwerte als einzelne ganzzahlige Klassenindizes vorliegen.

### Aufgabe 4: LeNet trainieren und Fehlerbilder untersuchen

Trainieren Sie das LeNet-Modell mit Batchgröße 32, höchstens zwölf Epochen und EarlyStopping mit Geduld 3. Zeichnen Sie Loss und Accuracy für Training und Validierung.

Bewerten Sie Accuracy und Macro-F1 auf dem Testdatensatz, erstellen Sie eine Konfusionsmatrix und visualisieren Sie bis zu sechs falsch klassifizierte Bilder mit wahrer Klasse, Vorhersage und maximaler Wahrscheinlichkeit.

In [ ]:
# Verwenden Sie das in Aufgabe 3 erzeugte lenet_modell.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

fruehstopp_lenet = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)
startzeit_lenet = time.perf_counter()
history_lenet = lenet_modell.fit(
    bilder_train,
    labels_train,
    validation_data=(bilder_val, labels_val),
    epochs=12,
    batch_size=32,
    callbacks=[fruehstopp_lenet],
    verbose=0,
)
trainingszeit_lenet = time.perf_counter() - startzeit_lenet

history_lenet_df = pd.DataFrame(history_lenet.history)
print("Epochen:", len(history_lenet_df))
print("Trainingszeit in Sekunden:", round(trainingszeit_lenet, 2))

plt.plot(history_lenet_df["loss"], label="Training")
plt.plot(history_lenet_df["val_loss"], label="Validierung")
plt.xlabel("Epoche")
plt.ylabel("Verlust")
plt.title("LeNet-Verlust")
plt.legend()
plt.show()

plt.plot(history_lenet_df["accuracy"], label="Training")
plt.plot(history_lenet_df["val_accuracy"], label="Validierung")
plt.xlabel("Epoche")
plt.ylabel("Accuracy")
plt.title("LeNet-Accuracy")
plt.legend()
plt.show()

lenet_wahrscheinlichkeiten = lenet_modell.predict(bilder_test, verbose=0)
lenet_prognose = np.argmax(lenet_wahrscheinlichkeiten, axis=1)
lenet_accuracy = accuracy_score(labels_test, lenet_prognose)
lenet_f1 = f1_score(labels_test, lenet_prognose, average="macro")
print("Test-Accuracy:", round(lenet_accuracy, 3))
print("Test-Macro-F1:", round(lenet_f1, 3))

ConfusionMatrixDisplay.from_predictions(labels_test, lenet_prognose)
plt.title("LeNet-Konfusionsmatrix")
plt.show()

fehler_indizes = np.flatnonzero(labels_test != lenet_prognose)[:6]
for index in fehler_indizes:
    plt.figure()
    plt.imshow(bilder_test[index, ..., 0], cmap="gray")
    plt.title(
        f"Wahr {labels_test[index]}, vorhergesagt {lenet_prognose[index]}, "
        f"Sicherheit {lenet_wahrscheinlichkeiten[index].max():.2f}"
    )
    plt.axis("off")
    plt.show()

> **Musterantwort und Interpretation**
>
> Überanpassung zeigt sich häufig, wenn der Trainingsverlust weiter sinkt oder die Trainings-Accuracy steigt, während sich der Validierungsverlust verschlechtert oder die Validierungs-Accuracy stagniert. EarlyStopping beendet das Training anhand der Validierung und stellt die Gewichte des besten beobachteten Zeitpunkts wieder her.

### Aufgabe 5: MobileNetV2 als eingefrorenen Merkmalsextraktor einsetzen

Bereiten Sie für Transfer Learning höchstens 700 Trainings-, 180 Validierungs- und 250 Testbilder vor. Skalieren Sie die Graustufenbilder mit TensorFlow auf 64-mal-64, wiederholen Sie den Kanal dreimal und wenden Sie `tf.keras.applications.mobilenet_v2.preprocess_input` auf Werte im Bereich 0 bis 255 an.

Laden Sie MobileNetV2 mit `include_top=False`. Versuchen Sie vortrainierte ImageNet-Gewichte zu verwenden. Falls der Download nicht verfügbar ist, soll der Code kontrolliert auf `weights=None` zurückfallen und dies deutlich melden. Frieren Sie die Basis ein und trainieren Sie nur einen kleinen Kopf aus GlobalAveragePooling und Dense für höchstens drei Epochen.

In [ ]:
VERSUCHE_VORTRAINIERTE_GEWICHTE = True

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def fuer_mobilenet_vorbereiten(bilder, maximale_anzahl):
    auswahl = np.asarray(bilder[:maximale_anzahl], dtype="float32")
    auswahl = auswahl[..., np.newaxis]
    # MobileNetV2 erwartet drei Farbkanäle und arbeitet auch mit kleinen 64er-Bildern.
    tensor = tf.image.resize(auswahl, size=(64, 64), method="bilinear")
    tensor = tf.repeat(tensor, repeats=3, axis=-1)
    # Die Originalwerte 0 bis 16 werden zunächst auf 0 bis 255 gebracht.
    tensor = tensor * (255.0 / 16.0)
    return tf.keras.applications.mobilenet_v2.preprocess_input(tensor)

transfer_train_x = fuer_mobilenet_vorbereiten(bilder_train_roh, 700)
transfer_val_x = fuer_mobilenet_vorbereiten(bilder_val_roh, 180)
transfer_test_x = fuer_mobilenet_vorbereiten(bilder_test_roh, 250)
transfer_train_y = labels_train[: len(transfer_train_x)]
transfer_val_y = labels_val[: len(transfer_val_x)]
transfer_test_y = labels_test[: len(transfer_test_x)]

print("Transfer-Formen:", transfer_train_x.shape, transfer_val_x.shape, transfer_test_x.shape)

verwendet_vortraining = False
try:
    gewichte = "imagenet" if VERSUCHE_VORTRAINIERTE_GEWICHTE else None
    mobilenet_basis = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights=gewichte,
        input_shape=(64, 64, 3),
    )
    verwendet_vortraining = gewichte is not None
except Exception as fehler:
    print("Vortrainierte Gewichte konnten nicht geladen werden:", type(fehler).__name__)
    print("Offline-Fallback mit zufälligen Gewichten wird verwendet.")
    mobilenet_basis = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights=None,
        input_shape=(64, 64, 3),
    )

# Erst einfrieren, dann den neuen Kopf aufbauen.
mobilenet_basis.trainable = False
transfer_modell = keras.Sequential(
    [
        keras.Input(shape=(64, 64, 3)),
        mobilenet_basis,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.20, seed=RANDOM_SEED),
        layers.Dense(10, activation="softmax"),
    ],
    name="mobilenet_transfer",
)
transfer_modell.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_transfer = transfer_modell.fit(
    transfer_train_x,
    transfer_train_y,
    validation_data=(transfer_val_x, transfer_val_y),
    epochs=3,
    batch_size=32,
    verbose=0,
)

print("Vortrainierte Gewichte verwendet:", verwendet_vortraining)
print("Trainierbare Parameter:", int(np.sum([np.prod(v.shape) for v in transfer_modell.trainable_weights])))
print("Nicht trainierbare Parameter:", int(np.sum([np.prod(v.shape) for v in transfer_modell.non_trainable_weights])))

> **Musterantwort und Interpretation**
>
> Bei zufälligen Basisgewichten enthält MobileNetV2 keine vorher gelernten allgemeinen Bildmerkmale. Wenn die Basis zugleich eingefroren bleibt, kann nur der Kopf zufällige Merkmalskarten kombinieren. Der Fallback hält den Notebookablauf technisch funktionsfähig, muss aber als Architekturdemonstration und nicht als belastbare Transfer-Learning-Bewertung dokumentiert werden.

### Aufgabe 6: Integrationsaufgabe: CNN, Transfer und klassische Baseline fair vergleichen

Trainieren Sie auf den identischen ursprünglichen Trainings- und Testbeobachtungen eine klassische Pipeline aus StandardScaler und logistischer Regression auf 64 Pixelmerkmalen.

Bewerten Sie die Pixelbaseline, das LeNet-Modell und das Transfermodell mit Accuracy und Macro-F1. Messen Sie für jeden Ansatz die Vorhersagezeit auf seinem jeweiligen vorbereiteten Testtensor und berichten Sie Gesamtparameter sowie trainierbare Parameter. Kennzeichnen Sie deutlich, ob echte vortrainierte Gewichte verfügbar waren. Formulieren Sie anschließend eine begründete Modellwahl für eine CPU-begrenzte Lernumgebung.

In [ ]:
# Für einen fairen Vergleich wird beim Transfermodell nur der verfügbare Transfer-Testanteil verwendet.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

X_pixel_train = bilder_train_roh.reshape(len(bilder_train_roh), -1) / 16.0
X_pixel_test = bilder_test_roh.reshape(len(bilder_test_roh), -1) / 16.0

pixel_baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2500, random_state=RANDOM_SEED),
)
pixel_baseline.fit(X_pixel_train, labels_train)

start = time.perf_counter()
pixel_pred = pixel_baseline.predict(X_pixel_test)
pixel_zeit = time.perf_counter() - start

start = time.perf_counter()
_ = lenet_modell.predict(bilder_test, verbose=0)
lenet_zeit = time.perf_counter() - start

start = time.perf_counter()
transfer_wahrscheinlichkeit = transfer_modell.predict(transfer_test_x, verbose=0)
transfer_zeit = time.perf_counter() - start
transfer_pred = np.argmax(transfer_wahrscheinlichkeit, axis=1)

# Eine scikit-learn-Parameterzahl kann aus allen Koeffizienten und Biaswerten gebildet werden.
logreg_schritt = pixel_baseline.named_steps["logisticregression"]
pixel_parameter = logreg_schritt.coef_.size + logreg_schritt.intercept_.size

vergleich_cnn = pd.DataFrame(
    [
        {
            "Modell": "Pixel-LogReg",
            "Accuracy": accuracy_score(labels_test, pixel_pred),
            "Macro_F1": f1_score(labels_test, pixel_pred, average="macro"),
            "Vorhersagezeit_s": pixel_zeit,
            "Parameter_gesamt": pixel_parameter,
            "Parameter_trainierbar": pixel_parameter,
            "Vortraining": False,
        },
        {
            "Modell": "Kleines LeNet",
            "Accuracy": accuracy_score(labels_test, lenet_prognose),
            "Macro_F1": f1_score(labels_test, lenet_prognose, average="macro"),
            "Vorhersagezeit_s": lenet_zeit,
            "Parameter_gesamt": lenet_modell.count_params(),
            "Parameter_trainierbar": int(np.sum([np.prod(v.shape) for v in lenet_modell.trainable_weights])),
            "Vortraining": False,
        },
        {
            "Modell": "MobileNetV2-Kopf",
            "Accuracy": accuracy_score(transfer_test_y, transfer_pred),
            "Macro_F1": f1_score(transfer_test_y, transfer_pred, average="macro"),
            "Vorhersagezeit_s": transfer_zeit,
            "Parameter_gesamt": transfer_modell.count_params(),
            "Parameter_trainierbar": int(np.sum([np.prod(v.shape) for v in transfer_modell.trainable_weights])),
            "Vortraining": verwendet_vortraining,
        },
    ]
)
display(vergleich_cnn.round(4))

> **Musterantwort und Interpretation**
>
> Die Entscheidung sollte die tatsächlich gemessenen Werte berücksichtigen. Auf 8-mal-8-Ziffern ist eine Pixel-LogReg oft überraschend stark, sehr schnell und leicht zu erklären. Ein kleines LeNet kann lokale Bildmuster lernen und bleibt noch überschaubar. MobileNetV2 ist für natürliche Farbfotos vortrainiert, deutlich größer und benötigt künstliches Hochskalieren; sein Transfernutzen kann bei winzigen Spezialbildern begrenzt sein. Ohne verfügbare ImageNet-Gewichte ist der Transfervergleich nur eine technische Demonstration.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?